In [1]:
from scipy.io import loadmat
import numpy as np
import pandas as pd

mat = loadmat("../data/sessionevents01.mat")  # sample file in data/

t = mat["t"]
data = mat["data"]
labels = mat["labels"]

print("t shape:", t.shape)
print("data shape:", data.shape)
print("labels shape:", labels.shape)

cols = ["session", "current", "last", "solo",
        "cardValue", "countP1", "countP2", "countP3"]
labels_df = pd.DataFrame(labels, columns=cols)
labels_df.head()


t shape: (501, 1)
data shape: (501, 32, 3, 249)
labels shape: (249, 8)


,session,current,last,solo,cardValue,countP1,countP2,countP3
0,1,3,0,1,0,0,0,0
1,1,1,3,1,0,0,0,1
2,1,2,1,1,0,1,0,1
3,1,2,0,1,0,1,1,1
4,1,3,2,1,0,1,2,1


In [2]:
n_time, n_chan, n_players, n_events = data.shape
print(n_time, n_chan, n_players, n_events)

X_list = []
y_list = []

for e in range(n_events):
    row = labels_df.iloc[e]
    for p in range(1, n_players + 1):
        eeg = data[:, :, p-1, e]        # (time, chan)
        eeg = np.nan_to_num(eeg)
        eeg = eeg.T                     # (chan, time)

        if p == row["last"]:
            role = "played"
        elif p == row["current"]:
            role = "next"
        else:
            role = "observer"

        X_list.append(eeg)
        y_list.append(role)

X = np.stack(X_list)
y = np.array(y_list)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("First 10 labels:", y[:10])


501 32 3 249
X shape: (747, 32, 501)
y shape: (747,)
First 10 labels: ['observer' 'observer' 'next' 'next' 'observer' 'played' 'played' 'next'
 'observer' 'observer']


In [3]:
n_time, n_chan, n_players, n_events = data.shape
print(n_time, n_chan, n_players, n_events)

X_list = []
y_list = []

for e in range(n_events):
    row = labels_df.iloc[e]
    for p in range(1, n_players + 1):
        eeg = data[:, :, p-1, e]        # (time, chan)
        eeg = np.nan_to_num(eeg)
        eeg = eeg.T                     # (chan, time)

        if p == row["last"]:
            role = "played"
        elif p == row["current"]:
            role = "next"
        else:
            role = "observer"

        X_list.append(eeg)
        y_list.append(role)

X = np.stack(X_list)
y = np.array(y_list)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("First 10 labels:", y[:10])


501 32 3 249
X shape: (747, 32, 501)
y shape: (747,)
First 10 labels: ['observer' 'observer' 'next' 'next' 'observer' 'played' 'played' 'next'
 'observer' 'observer']


In [4]:
from sklearn.model_selection import train_test_split

label_to_int = {"next": 0, "observer": 1, "played": 2}
y_int = np.array([label_to_int[l] for l in y])

X_dl = X[..., np.newaxis]   # (samples, chan, time, 1)
print("X_dl shape:", X_dl.shape)
print("y_int[:10]:", y_int[:10])

X_train, X_test, y_train, y_test = train_test_split(
    X_dl, y_int, test_size=0.2, random_state=42, stratify=y_int
)

print("Train:", X_train.shape, "Test:", X_test.shape)


X_dl shape: (747, 32, 501, 1)
y_int[:10]: [1 1 0 0 1 2 2 0 1 1]
Train: (597, 32, 501, 1) Test: (150, 32, 501, 1)


In [5]:
import tensorflow as tf
from tensorflow.keras import layers, models

input_shape = X_train.shape[1:]  # (32, 501, 1)

model = models.Sequential([
    layers.Conv2D(16, (1, 15), activation='relu', padding='same', input_shape=input_shape),
    layers.MaxPooling2D((1, 2)),

    layers.Conv2D(32, (1, 15), activation='relu', padding='same'),
    layers.MaxPooling2D((1, 2)),

    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(3, activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 32, 501, 16)       256       
                                                                 
 max_pooling2d (MaxPooling2D  (None, 32, 250, 16)      0         
 )                                                               
                                                                 
 conv2d_1 (Conv2D)           (None, 32, 250, 32)       7712      
                                                                 
 max_pooling2d_1 (MaxPooling  (None, 32, 125, 32)      0         
 2D)                                                             
                                                                 
 flatten (Flatten)           (None, 128000)            0         
                                                                 
 dense (Dense)               (None, 64)                8

In [6]:
history = model.fit(
    X_train, y_train,
    epochs=25,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)


Epoch 1/25
60/60 [==============================] - 16s 207ms/step - loss: 6.6003 - accuracy: 0.4151 - val_loss: 1.0563 - val_accuracy: 0.4750
Epoch 2/25
60/60 [==============================] - 11s 186ms/step - loss: 0.8573 - accuracy: 0.6289 - val_loss: 1.1020 - val_accuracy: 0.4250
Epoch 3/25
60/60 [==============================] - 11s 185ms/step - loss: 0.4698 - accuracy: 0.8008 - val_loss: 1.1668 - val_accuracy: 0.5000
Epoch 4/25
60/60 [==============================] - 11s 182ms/step - loss: 0.2024 - accuracy: 0.9392 - val_loss: 1.4316 - val_accuracy: 0.4417
Epoch 5/25
60/60 [==============================] - 11s 183ms/step - loss: 0.0872 - accuracy: 0.9665 - val_loss: 1.6938 - val_accuracy: 0.4500
Epoch 6/25
60/60 [==============================] - 11s 190ms/step - loss: 0.0618 - accuracy: 0.9811 - val_loss: 2.1004 - val_accuracy: 0.4583
Epoch 7/25
60/60 [==============================] - 11s 182ms/step - loss: 0.0891 - accuracy: 0.9790 - val_loss: 1.8950 - val_accuracy: 0.4250

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print("Test accuracy:", test_acc)

from sklearn.metrics import classification_report

y_pred = model.predict(X_test)
y_pred_classes = y_pred.argmax(axis=1)

print(classification_report(y_test, y_pred_classes,
                            target_names=["next", "observer", "played"]))


Test accuracy: 0.47999998927116394
5/5 [==============================] - 1s 131ms/step
              precision    recall  f1-score   support

        next       0.45      0.36      0.40        50
    observer       0.49      0.67      0.57        67
      played       0.50      0.27      0.35        33

    accuracy                           0.48       150
   macro avg       0.48      0.43      0.44       150
weighted avg       0.48      0.48      0.46       150



: 